# RT-IoT2022 - Data Preprocessing

In this notebook we prepare the dataset for anomaly detection. The main steps are:

- load and clean the dataset (index column and duplicates)
- separate normal and anomalous traffic
- create training, validation and test sets
- encode categorical features
- scale numerical features
- save the processed datasets

The training set contains only normal traffic, because the models must learn what normal behaviour looks like. Validation and test sets contain both normal and anomalous traffic, so that we can measure how well the models detect attacks.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df = pd.read_csv("../data/raw/RT_IOT2022.csv")
print("Dataset shape:", df.shape)

df = df.drop(columns=["Unnamed: 0"])
print("Dataset shape after removing index column:", df.shape)

Dataset shape: (123117, 85)
Dataset shape after removing index column: (123117, 84)


In [3]:
normal_classes = ["MQTT_Publish", "Thing_Speak", "Wipro_bulb"]

df["Label"] = df["Attack_type"].apply(
    lambda x: "Normal" if x in normal_classes else "Anomaly"
)

print(df["Label"].value_counts())

Label
Anomaly    110610
Normal      12507
Name: count, dtype: int64


In [4]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicated rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicated rows after removal:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)

Missing values: 0
Duplicated rows: 5195
Duplicated rows after removal: 0
Dataset shape after removing duplicates: (117922, 85)


We remove the 5195 duplicated rows **before** splitting the data. If we did it after, the same flow could end up both in the training set and in the test set, and the model would be evaluated on data it has already seen (data leakage).

After this step the dataset contains 117922 rows.

In [5]:
normal_df = df[df["Label"] == "Normal"].copy()
anomaly_df = df[df["Label"] == "Anomaly"].copy()

print("Normal samples:", len(normal_df))
print("Anomaly samples:", len(anomaly_df))

Normal samples: 12015
Anomaly samples: 105907


After removing duplicates we have 12015 normal samples and 105907 anomalous samples.

### Data split

The data is split in this way:

- **Normal traffic:** 60% training, 20% validation, 20% test
- **Anomalous traffic:** 50% validation, 50% test (no anomalies in training)

Both splits are stratified by 'Attack_type', so every traffic class keeps the same proportion in each set. We use 'random_state=42' to make the split reproducible.

In [6]:
normal_train, normal_temp = train_test_split(
    normal_df,
    test_size = 0.40,
    random_state=42,
    stratify=normal_df["Attack_type"]
)

normal_val, normal_test = train_test_split(
    normal_temp,
    test_size=0.50,
    random_state=42,
    stratify=normal_temp["Attack_type"]
)

print("Normal training samples:", len(normal_train))
print("Normal validation samples:", len(normal_val))
print("Normal test samples:", len(normal_test))

Normal training samples: 7209
Normal validation samples: 2403
Normal test samples: 2403


In [7]:
anomaly_val, anomaly_test = train_test_split(
    anomaly_df,
    test_size = 0.50,
    random_state = 42,
    stratify = anomaly_df["Attack_type"]
)

print("Anomalies in validation:", len(anomaly_val))
print("Anomalies in test:", len(anomaly_test))

Anomalies in validation: 52953
Anomalies in test: 52954


In [8]:
val_df = pd.concat([normal_val, anomaly_val])
test_df = pd.concat([normal_test, anomaly_test])

val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

normal_train = normal_train.reset_index(drop=True)

print("Training set:", normal_train.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)

print("\nValidation labels:")
print(val_df["Label"].value_counts())

print("\nTest labels:")
print(test_df["Label"].value_counts())

Training set: (7209, 85)
Validation set: (55356, 85)
Test set: (55357, 85)

Validation labels:
Label
Anomaly    52953
Normal      2403
Name: count, dtype: int64

Test labels:
Label
Anomaly    52954
Normal      2403
Name: count, dtype: int64


The final sets are:

| Set | Normal | Anomalous | Total |
|---|---|---|---|
| Training | 7209 | 0 | 7209 |
| Validation | 2403 | 52953 | 55356 |
| Test | 2403 | 52954 | 55357 |

The validation set is used to choose hyperparameters and thresholds. The test set is used only once, at the end, to evaluate the final models.

About 96% of the test set is anomalous traffic. For this reason accuracy would be misleading, and we use Balanced Accuracy as the main metric.

In [9]:
feature_columns = [
    column for column in df.columns
    if column not in ["Attack_type", "Label"]
]

X_train = normal_train[feature_columns].copy()
X_val = val_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

train_metadata = normal_train[["Attack_type", "Label"]].copy()
val_metadata = val_df[["Attack_type", "Label"]].copy()
test_metadata = test_df[["Attack_type", "Label"]].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (7209, 83)
X_val: (55356, 83)
X_test: (55357, 83)


'Attack_type' and 'Label' are removed from the input features and saved separately as metadata. They are never given to the models: they are used only to evaluate the results.

In [10]:
numeric_columns = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))

print("\nCategorical features:")
print(categorical_columns)

Numeric features: 81
Categorical features: 2

Categorical features:
['proto', 'service']


In [11]:
print(
    "Infinite values in training:",
    np.isinf(X_train[numeric_columns]).sum().sum()
)

print(
    "Infinite values in validation:",
    np.isinf(X_val[numeric_columns]).sum().sum()
)

print(
    "Infinite values in test:",
    np.isinf(X_test[numeric_columns]).sum().sum()
)

Infinite values in training: 0
Infinite values in validation: 0
Infinite values in test: 0


### Encoding and Scaling

- The categorical features ('proto' and 'service') are converted with **one-hot encoding**.
- The numerical features are standardized with **StandardScaler** (mean 0, standard deviation 1), so that features with large values do not dominate the others.

Both the encoder and the scaler are fitted **only on the training set** and then applied to validation and test.

In [12]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(X_train[categorical_columns])
X_train_cat = encoder.transform(X_train[categorical_columns])
X_val_cat = encoder.transform(X_val[categorical_columns])
X_test_cat = encoder.transform(X_test[categorical_columns])

categorical_feature_names = encoder.get_feature_names_out(
    categorical_columns
)

print(categorical_feature_names)

['proto_icmp' 'proto_tcp' 'proto_udp' 'service_-' 'service_dhcp'
 'service_dns' 'service_http' 'service_irc' 'service_mqtt' 'service_ntp'
 'service_ssl']


In [13]:
scaler = StandardScaler()
scaler.fit(X_train[numeric_columns])
X_train_num = scaler.transform(X_train[numeric_columns])
X_val_num = scaler.transform(X_val[numeric_columns])
X_test_num = scaler.transform(X_test[numeric_columns])

In [14]:
final_columns = numeric_columns +list(categorical_feature_names)

X_train_processed = pd.DataFrame(
    np.hstack([X_train_num, X_train_cat]),
    columns = final_columns
)

X_val_processed = pd.DataFrame(
    np.hstack([X_val_num, X_val_cat]),
    columns = final_columns
)

X_test_processed = pd.DataFrame(
    np.hstack([X_test_num, X_test_cat]),
    columns = final_columns
)

print("Processed training set:", X_train_processed.shape)
print("Processed validation set:", X_val_processed.shape)
print("Processed test set:", X_test_processed.shape)

Processed training set: (7209, 92)
Processed validation set: (55356, 92)
Processed test set: (55357, 92)


After preprocessing, each sample has 92 features: 81 scaled numerical features and 11 one-hot encoded categorical features. We also check that there are no missing or infinite values.

In [15]:
print(
    "Missing values in training:",
    X_train_processed.isnull().sum().sum()
)

print(
    "Missing values in validation:",
    X_val_processed.isnull().sum().sum()
)

print(
    "Missing values in test:",
    X_test_processed.isnull().sum().sum() 
)

print(
    "Infinite values in processed training:",
    np.isinf(X_train_processed.to_numpy()).sum()
)

print(
    "Infinite values in processed validation:",
    np.isinf(X_val_processed.to_numpy()).sum()
)

print(
   "Infinite values in processed test:",
   np.isinf(X_test_processed.to_numpy()).sum()
)

Missing values in training: 0
Missing values in validation: 0
Missing values in test: 0
Infinite values in processed training: 0
Infinite values in processed validation: 0
Infinite values in processed test: 0


In [16]:
X_train_processed.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_val_processed.to_csv(
    "../data/processed/X_val.csv",
    index=False
)

X_test_processed.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

train_metadata.to_csv(
    "../data/processed/train_metadata.csv",
    index=False
)

val_metadata.to_csv(
    "../data/processed/val_metadata.csv",
    index=False
)

test_metadata.to_csv(
    "../data/processed/test_metadata.csv",
    index=False
)

In [17]:
print("Preprocessing completed successfully")

print("Training:")
print(X_train_processed.shape)

print("\nvalidation:")
print(X_val_processed.shape)

print("\nTest:")
print(X_test_processed.shape)

Preprocessing completed successfully
Training:
(7209, 92)

validation:
(55356, 92)

Test:
(55357, 92)


## Conclusions

Thanks to this phase, the dataset is now ready for the anomaly detection models:

- duplicates are removed before the split
- the training set contains only normal traffic (7209 samples)
- validation and test contain both normal and anomalous traffic
- encoder and scaler are fitted only on the training set, to avoid data leakage.

The processed datasets are saved in 'data/processed/' and are used by all the models, so each one is trained and evaluated on exactly the same data.